In [1]:
# Principal:    bi_user
# Role:         viewer_gold
# Catalog Role: viewer_catalog_role
# Привилегии:   NAMESPACE_LIST (catalog)
#               gold: TABLE_LIST, TABLE_READ_DATA
#
# Матрица доступов:
#   bronze: НЕТ ДОСТУПА
#   silver: НЕТ ДОСТУПА
#   gold:   только чтение
#
# FORBIDDEN: SELECT bronze, SELECT silver, INSERT gold, CREATE gold, DROP gold

In [2]:
import os
from pyspark.sql import SparkSession

client_id = os.environ["BI_USER_CLIENT_ID"]
client_secret = os.environ["BI_USER_CLIENT_SECRET"]
credential = f"{client_id}:{client_secret}"

spark = SparkSession.builder \
    .appName("lakehouse-bi-user-rbac") \
    .config("spark.sql.catalog.lakehouse.credential", credential) \
    .getOrCreate()

spark

In [3]:
spark.sql("SHOW CATALOGS").show(truncate=False)
spark.sql("SHOW TABLES IN lakehouse.gold").show(truncate=False)

+-------------+
|catalog      |
+-------------+
|lakehouse    |
|spark_catalog|
+-------------+

+---------+----------------------+-----------+
|namespace|tableName             |isTemporary|
+---------+----------------------+-----------+
|gold     |mart_sales_by_category|false      |
|gold     |mart_top_customers    |false      |
+---------+----------------------+-----------+



In [4]:
print("[ALLOWED] TABLE_LIST — lakehouse.gold")
spark.sql("SHOW TABLES IN lakehouse.gold").show(truncate=False)

[ALLOWED] TABLE_LIST — lakehouse.gold
+---------+----------------------+-----------+
|namespace|tableName             |isTemporary|
+---------+----------------------+-----------+
|gold     |mart_sales_by_category|false      |
|gold     |mart_top_customers    |false      |
+---------+----------------------+-----------+



In [5]:
print("[ALLOWED] TABLE_READ_DATA — lakehouse.gold.mart_sales_by_category")
spark.sql("""
    SELECT
        category_name,
        month,
        total_revenue,
        order_count,
        avg_check
    FROM lakehouse.gold.mart_sales_by_category
    LIMIT 3
""").show(truncate=False)

[ALLOWED] TABLE_READ_DATA — lakehouse.gold.mart_sales_by_category
+-------------+-------+-------------+-----------+---------+
|category_name|month  |total_revenue|order_count|avg_check|
+-------------+-------+-------------+-----------+---------+
|Books        |2025-05|9691.97      |106        |91.43    |
|Clothing     |2025-05|30004.05     |86         |348.88   |
|Electronics  |2025-05|249805.97    |104        |2401.98  |
+-------------+-------+-------------+-----------+---------+



In [6]:
print("[ALLOWED] TABLE_FULL_METADATA — lakehouse.gold.mart_sales_by_category")
spark.sql("DESCRIBE TABLE EXTENDED lakehouse.gold.mart_sales_by_category").show(truncate=False)

[ALLOWED] TABLE_FULL_METADATA — lakehouse.gold.mart_sales_by_category
+-----------------------------+----------------------------------------------------------------+-------+
|col_name                     |data_type                                                       |comment|
+-----------------------------+----------------------------------------------------------------+-------+
|category_name                |string                                                          |NULL   |
|month                        |string                                                          |NULL   |
|total_revenue                |decimal(14,2)                                                   |NULL   |
|order_count                  |bigint                                                          |NULL   |
|avg_check                    |decimal(12,2)                                                   |NULL   |
|                             |                                                           

In [7]:
print("[FORBIDDEN] SELECT из bronze (bronze закрыт)")
try:
    spark.sql("""
        SELECT
            id,
            name
        FROM lakehouse.bronze.raw_categories
        LIMIT 1
    """).show(truncate=False)
    print("[ПРОВАЛ] Доступ разрешён — ожидалось запрещено")
except Exception as e:
    short = "\n".join(str(e).split("\n")[:3])
    print(f"[ОЖИДАЕМО] Доступ запрещён: {short}")
    print()

print("[FORBIDDEN] SELECT из silver (silver закрыт)")
try:
    spark.sql("""
        SELECT
            id,
            name
        FROM lakehouse.silver.customers
        LIMIT 1
    """).show(truncate=False)
    print("[ПРОВАЛ] Доступ разрешён — ожидалось запрещено")
except Exception as e:
    short = "\n".join(str(e).split("\n")[:3])
    print(f"[ОЖИДАЕМО] Доступ запрещён: {short}")
    print()

print("[FORBIDDEN] INSERT в gold.mart_top_customers (только чтение)")
try:
    spark.sql("""
        INSERT INTO lakehouse.gold.mart_top_customers
        VALUES (99999, 'test', 1, CAST(1 AS BIGINT), CAST(100.00 AS DECIMAL(14,2)), 'Low')
    """)
    print("[ПРОВАЛ] Доступ разрешён — ожидалось запрещено")
except Exception as e:
    short = "\n".join(str(e).split("\n")[:3])
    print(f"[ОЖИДАЕМО] Доступ запрещён: {short}")
    print()

print("[FORBIDDEN] CREATE TABLE в gold (нет TABLE_CREATE)")
try:
    spark.sql("""
        CREATE TABLE lakehouse.gold._test_probe (
            id   INT,
            name STRING
        ) USING iceberg
    """)
    print("[ПРОВАЛ] Доступ разрешён — ожидалось запрещено")
except Exception as e:
    short = "\n".join(str(e).split("\n")[:3])
    print(f"[ОЖИДАЕМО] Доступ запрещён: {short}")
    print()

print("[FORBIDDEN] DROP TABLE gold.mart_sales_by_category (нет TABLE_DROP)")
try:
    spark.sql("DROP TABLE lakehouse.gold.mart_sales_by_category")
    print("[ПРОВАЛ] Доступ разрешён — ожидалось запрещено")
except Exception as e:
    short = "\n".join(str(e).split("\n")[:3])
    print(f"[ОЖИДАЕМО] Доступ запрещён: {short}")
    print()

[FORBIDDEN] SELECT из bronze (bronze закрыт)
[ОЖИДАЕМО] Доступ запрещён: An error occurred while calling o43.sql.
: org.apache.iceberg.exceptions.ForbiddenException: Forbidden: Principal 'bi_user' with activated PrincipalRoles '[viewer_gold]' and activated grants via '[viewer_catalog_role, viewer_gold]' is not authorized for op LOAD_TABLE
	at org.apache.iceberg.rest.ErrorHandlers$DefaultErrorHandler.accept(ErrorHandlers.java:238)

[FORBIDDEN] SELECT из silver (silver закрыт)
[ОЖИДАЕМО] Доступ запрещён: An error occurred while calling o43.sql.
: org.apache.iceberg.exceptions.ForbiddenException: Forbidden: Principal 'bi_user' with activated PrincipalRoles '[viewer_gold]' and activated grants via '[viewer_catalog_role, viewer_gold]' is not authorized for op LOAD_TABLE
	at org.apache.iceberg.rest.ErrorHandlers$DefaultErrorHandler.accept(ErrorHandlers.java:238)

[FORBIDDEN] INSERT в gold.mart_top_customers (только чтение)
[ОЖИДАЕМО] Доступ запрещён: An error occurred while calling o43.sql.
